# FSE Puglia redesign – survey analysis
Small sample (n≈41): everything here is **descriptive and directional**, not statistically conclusive.
Always report the *n* next to every number.

Sections: 0 Setup · 1 Sample profile · 2 Adoption · 3 Usability (Likert) · 4 Dependence on others · 5 Failures & channels · 6 Time & anxiety · 7 Accessibility · 8 Open-text coding · 9 Export

## 0. Setup & cleaning

In [24]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

PATH = "fse_questionnaire.csv"   # change if needed
raw = pd.read_csv(PATH)

# Rename the long questions to short names (by position, so the order of the form must not change)
short = ["timestamp","age","region","digital_skill","difficulty",
         "used_fse","experience","frequency","top_service","useful_features",
         "autonomy","failed_op","failed_why","non_use_why",
         "booking_channel","non_online_why",
         "l_easy","l_confident","l_clear_steps","l_icons","l_info","anxiety",
         "booking_time","guided_someone","guided_why","guide_ease","helped_alone_next",
         "booked_for_other","booked_for_why","satisfaction","wishes"]
assert len(short) == raw.shape[1]
df = raw.copy(); df.columns = short
df["id"] = df.index

# Cleaning
df = df.apply(lambda s: s.str.strip() if s.dtype == "object" else s)
df["region"] = df["region"].replace({"Altro": "Other"})
# '< 60 anni' was a typo in the form options: it means '> 60 anni'
df["age"] = df["age"].replace({"< 60 anni": "> 60 anni"})

# Keep only the two regions under comparison (drops the 5 "Other" respondents)
df = df[df["region"].isin(["Puglia", "Emilia-Romagna"])].copy()
print(df["region"].value_counts())
print(df["age"].value_counts())

likert = ["l_easy","l_confident","l_clear_steps","l_icons","l_info"]
df["anxiety_rev"] = 6 - df["anxiety"]                       # reverse-coded: higher = better
df["usability_idx"] = df[likert + ["anxiety_rev"]].mean(axis=1)   # composite 1-5
time_order = ["meno di 5 minuti","5-10 minuti","10-20","non riesco a completare la prenotazione"]
df["booking_time"] = pd.Categorical(df["booking_time"], time_order, ordered=True)
df["time_code"] = df["booking_time"].cat.codes.replace(-1, np.nan)

def explode_multi(col):
    # multi-select answers are separated by ';'
    return (df[["id","region",col]].dropna().assign(**{col: lambda d: d[col].str.split(";")})
            .explode(col).reset_index(drop=True))

# Handy: main comparison groups. Puglia = client, Emilia-Romagna = competitor, Other = reference only
GROUPS = ["Puglia","Emilia-Romagna","Other"]
df.shape

region
Puglia            24
Emilia-Romagna    12
Name: count, dtype: int64
age
18 - 35 anni    18
36 - 60 anni    11
> 60 anni        7
Name: count, dtype: int64


(36, 35)

## 1. Sample profile
Check whether the regions are comparable before comparing scores.

In [47]:
for col in ["age","digital_skill"]:
    print(f"\n== {col} ==")
    print(pd.crosstab(df[col], df["region"], margins=True))
print("\nMean digital skill by region:"); print(df.groupby("region")["digital_skill"].agg(["mean","count"]).round(2))


== age ==
region        Emilia-Romagna  Puglia  All
age                                      
18 - 35 anni               6      12   18
36 - 60 anni               4       7   11
> 60 anni                  2       5    7
All                       12      24   36

== digital_skill ==
region         Emilia-Romagna  Puglia  All
digital_skill                             
3                           4      12   16
4                           4       6   10
5                           4       6   10
All                        12      24   36

Mean digital skill by region:
                mean  count
region                     
Emilia-Romagna  4.00     12
Puglia          3.75     24


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

age_pct   = pd.crosstab(df["region"], df["age"], normalize="index").mul(100).round(1)
skill_pct = pd.crosstab(df["region"], df["digital_skill"], normalize="index").mul(100).round(1)
n_by_region = df["region"].value_counts()

AGE_ORDER = ["18 - 35 anni", "36 - 60 anni", "> 60 anni"]
AGE_COLORS = dict(zip(AGE_ORDER, ["#fdd0a2", "#fd8d3c", "#a63603"]))      # light -> dark orange

SKILL_ORDER = [1, 2, 3, 4, 5]
SKILL_COLORS = dict(zip(SKILL_ORDER, ["#c6dbef", "#9ecae1", "#6baed6", "#3182bd", "#08519c"]))  # light -> dark blue

# Region labels with n, e.g. "Puglia (n=24)"
labels = {r: f"{r} (n={n_by_region[r]})" for r in age_pct.index}

fig = make_subplots(rows=1, cols=2, subplot_titles=("Age by region (%)",
                                                    "Digital skill (1-5) by region (%)"))

for col in age_pct.columns:
    fig.add_trace(go.Bar(name=str(col), x=[labels[r] for r in age_pct.index], y=age_pct[col],
                         legendgroup="age", legendgrouptitle_text="Age",
                         marker_color=AGE_COLORS[col],
                         hovertemplate="%{x}<br>" + str(col) + ": %{y:.1f}%<extra></extra>"),
                  row=1, col=1)

for col in skill_pct.columns:
    fig.add_trace(go.Bar(name=f"skill {col}", x=[labels[r] for r in skill_pct.index], y=skill_pct[col],
                         legendgroup="skill", legendgrouptitle_text="Skill",
                         marker_color=SKILL_COLORS[col],
                         hovertemplate="%{x}<br>skill " + str(col) + ": %{y:.1f}%<extra></extra>"),
                  row=1, col=2)

fig.update_layout(barmode="stack", height=420, width=1000, template="simple_white",
                  legend=dict(font=dict(size=10)))
fig.update_yaxes(title_text="%", range=[0, 100])
fig.show()

## 2. Adoption funnel

In [27]:
adopt = pd.crosstab(df["region"], df["used_fse"], margins=True)
adopt["% used"] = (adopt.get("Si",0) / adopt["All"] * 100).round(0)
display(adopt)
display(pd.crosstab(df["age"], df["used_fse"], margins=True))
display(pd.crosstab(df["frequency"].fillna("(non-user)"), df["region"]))
display(pd.crosstab(df["top_service"].fillna("(non-user)"), df["region"]))

used_fse,No,Si,All,% used
region,,,,
Emilia-Romagna,2,10,12,83.0
Puglia,9,15,24,62.0
All,11,25,36,69.0


used_fse,No,Si,All
age,,,
18 - 35 anni,5,13,18
36 - 60 anni,1,10,11
> 60 anni,5,2,7
All,11,25,36


region,Emilia-Romagna,Puglia
frequency,,
(non-user),2,9
Circa una volta al mese,5,2
Più volte al mese,1,1
Qualche volta all'anno,4,12


region,Emilia-Romagna,Puglia
top_service,,
(non-user),2,9
Consultazione diario vaccinazioni,0,3
Consultazione referti,7,7
Prenotazione di una visita,1,5
Scaricare ricette da mostrare in farmacia per acquistare medicinali,1,0
Stampa prescrizioni e referti,1,0


In [51]:
import plotly.graph_objects as go

REGION_COLORS = {"Puglia": "#2a9d8f", "Emilia-Romagna": "#8e6bb0"}
steps = ["Respondents", "Have used the FSE", "Use it monthly or more", "Main use is booking visits"]

fig = go.Figure()
for r in ["Puglia", "Emilia-Romagna"]:
    d = df[df["region"] == r]
    vals = [
        len(d),
        (d["used_fse"] == "Si").sum(),
        d["frequency"].isin(["Circa una volta al mese", "Più volte al mese"]).sum(),
        (d["top_service"] == "Prenotazione di una visita").sum(),
    ]
    pct = [v / vals[0] * 100 for v in vals]
    fig.add_trace(go.Funnel(
        name=f"{r} (n={vals[0]})", y=steps, x=pct,
        text=[f"{v} people ({p:.0f}%)" for v, p in zip(vals, pct)], textinfo="text",
        marker_color=REGION_COLORS[r],
        hovertemplate="%{y}<br>%{text}<extra></extra>",
    ))

fig.update_layout(funnelmode="group", height=450, width=950, template="simple_white",
                  title="Adoption funnel (% of respondents in each region)",
                  legend=dict(orientation="h", y=-0.1))
fig.show()

In [52]:
REASON_GROUPS = {
    "Non sapevo che esistesse": "Awareness: didn't know it exists",
    "Non so come accedere": "Access: don't know how to log in",
    "Lo trovo complicato": "Usability: find it complicated",
    "Preferisco altri metodi": "Preference / no need",
    "Non ne ho avuto la necessità": "Preference / no need",
    "Non c'è mai stata necessità": "Preference / no need",
}

nu = explode_multi("non_use_why")
nu["group"] = nu["non_use_why"].map(REASON_GROUPS).fillna("Other")
counts = pd.crosstab(nu["group"], nu["region"]).reindex(columns=["Puglia", "Emilia-Romagna"], fill_value=0)
counts = counts.loc[counts.sum(axis=1).sort_values().index]      # largest at the top

n_nonusers = df[df["used_fse"] == "No"].groupby("region").size()

fig = go.Figure()
for r in ["Puglia", "Emilia-Romagna"]:
    fig.add_trace(go.Bar(
        y=counts.index, x=counts[r], orientation="h",
        name=f"{r} ({n_nonusers.get(r, 0)} non-users)", marker_color=REGION_COLORS[r],
        text=counts[r], textposition="outside",
    ))
fig.update_layout(barmode="group", height=420, width=950, template="simple_white",
                  title="Reasons for not using the FSE (number of people, multiple answers allowed)",
                  xaxis_title="People", margin=dict(l=260), legend=dict(orientation="h", y=-0.2))
fig.show()

## 3. Usability scores (Likert 1-5), by region
These questions were answered only by people who book online (≈32 of 41), so n per group is smaller. Anxiety is reverse-coded in the composite index.

In [29]:
items = likert + ["anxiety","satisfaction","usability_idx"]
summary = df[df["region"].isin(GROUPS)].groupby("region")[items].agg(["mean","median","count"]).round(2)
display(summary.T.unstack(level=1) if False else summary)

l_easy              l_confident              l_clear_steps  \
                 mean median count        mean median count          mean   
region                                                                      
Emilia-Romagna   2.90    3.0    10        3.40    3.0    10          3.50   
Puglia           3.22    3.0    18        3.44    4.0    18          3.44   

                            l_icons  ... l_info anxiety               \
               median count    mean  ...  count    mean median count   
region                               ...                               
Emilia-Romagna    3.5    10    3.60  ...     10    2.27    2.0    11   
Puglia            4.0    18    3.22  ...     18    2.50    2.5    18   

               satisfaction              usability_idx               
                       mean median count          mean median count  
region                                                               
Emilia-Romagna         3.00    3.0    11          3.42   3.67    11  
Puglia                 3.21    3.0    19          3.32   3.42    18  

[2 rows x 24 columns]

In [30]:
# Compact mean table + gap Puglia vs Emilia-Romagna
means = df.groupby("region")[items].mean().round(2).T
means["gap Puglia-ER"] = (means["Puglia"] - means["Emilia-Romagna"]).round(2)
display(means)

region,Emilia-Romagna,Puglia,gap Puglia-ER
l_easy,2.90,3.22,0.32
l_confident,3.40,3.44,0.04
l_clear_steps,3.50,3.44,-0.06
l_icons,3.60,3.22,-0.38
l_info,3.10,3.11,0.01
anxiety,2.27,2.50,0.23
satisfaction,3.00,3.21,0.21
usability_idx,3.42,3.32,-0.10


In [46]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COLORS = {"neg": "#c0504d", "neu": "#d9d9d9", "pos": "#4f81bd"}
items = likert + ["satisfaction"]
regions = ["Puglia", "Emilia-Romagna"]

def likert_table(data):
    rows = []
    for item in items:
        s = data[item].dropna()
        n = len(s)
        rows.append((item, n, (s <= 2).mean() * 100, (s == 3).mean() * 100, (s >= 4).mean() * 100))
    return pd.DataFrame(rows, columns=["item", "n", "neg", "neu", "pos"])

fig = make_subplots(rows=1, cols=2, subplot_titles=regions, horizontal_spacing=0.18)

for col, g in enumerate(regions, start=1):
    t = likert_table(df[df["region"] == g])
    y = [f"{i} (n={n})" for i, n in zip(t["item"], t["n"])]

    # Neutral is centered on zero; negative extends left, positive extends right
    fig.add_trace(go.Bar(y=y, x=t["neu"], base=-t["neu"] / 2, orientation="h",
                         name="3 (neutral)", marker_color=COLORS["neu"],
                         legendgroup="neu", showlegend=(col == 1),
                         text=t["neu"].round(0).astype(int).astype(str) + "%", textposition="inside",
                         hovertemplate="%{y}<br>neutral: %{x:.0f}%<extra></extra>"), row=1, col=col)
    fig.add_trace(go.Bar(y=y, x=-t["neg"], base=-t["neu"] / 2, orientation="h",
                         name="1-2 (negative)", marker_color=COLORS["neg"],
                         legendgroup="neg", showlegend=(col == 1),
                         text=t["neg"].round(0).astype(int).astype(str) + "%", textposition="inside",
                         hovertemplate="%{y}<br>negative: %{customdata:.0f}%<extra></extra>",
                         customdata=t["neg"]), row=1, col=col)
    fig.add_trace(go.Bar(y=y, x=t["pos"], base=t["neu"] / 2, orientation="h",
                         name="4-5 (positive)", marker_color=COLORS["pos"],
                         legendgroup="pos", showlegend=(col == 1),
                         text=t["pos"].round(0).astype(int).astype(str) + "%", textposition="inside",
                         hovertemplate="%{y}<br>positive: %{x:.0f}%<extra></extra>"), row=1, col=col)

    fig.update_yaxes(autorange="reversed", row=1, col=col)               # first item on top
    fig.add_vline(x=0, line_width=1, line_color="black", row=1, col=col)

ticks = [-60, -40, -20, 0, 20, 40, 60]
fig.update_xaxes(range=[-70, 70], tickvals=ticks, ticktext=[abs(v) for v in ticks], title_text="% of respondents")
fig.update_layout(barmode="overlay", height=450, width=1100, template="simple_white",
                  legend=dict(orientation="h", y=-0.2))
fig.show()

In [32]:
# Which item is weakest in Puglia?  (lowest mean = top design priority)
print(df[df.region=="Puglia"][likert].mean().sort_values().round(2))
# Anxiety vs satisfaction etc. is in section 6

l_info           3.11
l_easy           3.22
l_icons          3.22
l_confident      3.44
l_clear_steps    3.44
dtype: float64


## 4. Dependence on others

In [33]:
display(pd.crosstab(df["autonomy"].fillna("(non-user)"), df["age"]))
display(pd.crosstab(df["autonomy"].fillna("(non-user)"), df["digital_skill"]))
print("\nGuided someone:"); display(pd.crosstab(df["guided_someone"], df["region"]))
print("\nHelped person managed alone next time:"); display(pd.crosstab(df["helped_alone_next"].dropna(), df["region"]))
print("\nBooked for a relative/friend:"); display(pd.crosstab(df["booked_for_other"].dropna(), df["region"]))
display(df.loc[df.booked_for_why.notna(), ["region","booked_for_why"]])
display(df.loc[df.guided_why.notna(), ["region","guided_why","guide_ease"]])

age,18 - 35 anni,36 - 60 anni,> 60 anni
autonomy,,,
(non-user),5,1,5
A volte con aiuto,0,4,0
In autonomia,13,6,2


digital_skill,3,4,5
autonomy,,,
(non-user),5,3,3
A volte con aiuto,4,0,0
In autonomia,7,7,7



Guided someone:


region,Emilia-Romagna,Puglia
guided_someone,,
No,11,16
Si,1,7



Helped person managed alone next time:


region,Emilia-Romagna,Puglia
helped_alone_next,,
"No, mi è stato chiesto nuovamente aiuto",1,6
Si,2,5



Booked for a relative/friend:


region,Emilia-Romagna,Puglia
booked_for_other,,
No,9,15
Si,1,6


,region,booked_for_why
3,Puglia,Età avanzata
6,Emilia-Romagna,Language barrier
7,Emilia-Romagna,Mancanza di identità elettronica (SPID/CIE)
9,Emilia-Romagna,Età avanzata
18,Puglia,Difficoltà digitali
21,Puglia,Età avanzata
27,Puglia,Età avanzata
29,Puglia,Difficoltà digitali
33,Puglia,Età avanzata


,region,guided_why,guide_ease
3,Puglia,Perché non riuscivano a farlo,4.0
5,Emilia-Romagna,Perché offrivo supporto in uno sportello di facilitazione digitale,NaN
18,Puglia,La persona utilizza il servizio raramente ed ha scarse competenze digitali,3.0
29,Puglia,Analfabetismo digitale,5.0
36,Puglia,Perché non si possedevano le competenze adatte.,5.0


## 5. Failure points & booking channels

In [34]:
display(df.loc[df.failed_why.notna(), ["region","failed_why"]])
print("Channel by region:"); display(pd.crosstab(df["booking_channel"], df["region"], margins=True))
nonline = explode_multi("non_online_why")
display(pd.crosstab(nonline["non_online_why"], nonline["region"], margins=True).sort_values("All", ascending=False))

,region,failed_why
0,Emilia-Romagna,Per prenotare un'ecografia non mi mostra le disponibilità e non mi da alternative. Quindi rimango nella pagina della...
18,Puglia,Mancanza di date disponibili
22,Emilia-Romagna,"Perché a volte nel tentativo di fare una prenotazione on line,il programma non va avanti ma continua sempre a ritorn..."
24,Emilia-Romagna,Non sono riuscito a trovare il libretto dei vaccini
29,Puglia,Bug del sito
33,Puglia,Si


Channel by region:


region,Emilia-Romagna,Puglia,All
booking_channel,,,
Di persona allo sportello,5,9,14
Tramite chiamata,3,9,12
Tramite il portale online,4,5,9
All,12,23,35


region,Emilia-Romagna,Puglia,All
non_online_why,,,
All,9,23,32
È più veloce di persona o per telefono,5,13,18
Non so come fare,2,3,5
Le disponibilità delle strutture non sono chiare sul portale,0,3,3
Grazie a Dio non ho bisogno di visite specialistiche,0,1,1
Ho difficoltà nell'utilizzo,0,1,1
In alcuni casi è necessaria l'interazione,0,1,1
Mi sento più sicura al cup,1,0,1
Non L ho mai utilizzato,0,1,1


## 6. Time on task, anxiety, satisfaction

In [35]:
display(pd.crosstab(df["booking_time"], df["region"]))
display(df.groupby("digital_skill")[["time_code","anxiety","satisfaction"]].agg(["mean","count"]).round(2))

def sp(a, b):
    d = df[[a,b]].dropna()
    r, p = spearmanr(d[a], d[b])
    return pd.Series({"n":len(d), "rho":round(r,2), "p":round(p,3)})
pairs = [("anxiety","satisfaction"),("anxiety","l_confident"),("digital_skill","anxiety"),
         ("digital_skill","time_code"),("digital_skill","satisfaction"),("l_easy","satisfaction")]
display(pd.DataFrame({f"{a} ~ {b}": sp(a,b) for a,b in pairs}).T)
# With n this small, read rho as direction/strength only; p-values are indicative.

region,Emilia-Romagna,Puglia
booking_time,,
meno di 5 minuti,1,4
5-10 minuti,4,6
10-20,5,6
non riesco a completare la prenotazione,1,1


time_code       anxiety       satisfaction      
                   mean count    mean count         mean count
digital_skill                                                 
3                  1.50    12    2.62    13         3.14    14
4                  1.43     7    2.57     7         3.38     8
5                  1.11     9    2.00     9         2.88     8

,n,rho,p
anxiety ~ satisfaction,27.0,-0.12,0.552
anxiety ~ l_confident,28.0,-0.42,0.025
digital_skill ~ anxiety,29.0,-0.19,0.328
digital_skill ~ time_code,28.0,-0.18,0.347
digital_skill ~ satisfaction,30.0,-0.02,0.917
l_easy ~ satisfaction,26.0,0.70,0.000


In [53]:
import plotly.graph_objects as go

labels = {
    "l_easy": "Booking is easy",
    "l_confident": "I feel confident, no fear of mistakes",
    "l_clear_steps": "Steps are clear",
    "l_icons": "Buttons and icons are intuitive",
    "l_info": "Information is easy to understand",
    "anxiety_rev": "Low anxiety (reversed)",   # reversed so that higher = better, like the others
    "digital_skill": "Digital skill",
}

rows = []
for col, name in labels.items():
    d = df[[col, "satisfaction"]].dropna()
    rho, p = spearmanr(d[col], d["satisfaction"])
    rows.append((name, rho, len(d)))

drivers = pd.DataFrame(rows, columns=["item", "rho", "n"]).sort_values("rho")

fig = go.Figure(go.Bar(
    x=drivers["rho"], y=[f"{i} (n={n})" for i, n in zip(drivers["item"], drivers["n"])],
    orientation="h",
    marker_color=["#4f81bd" if r >= 0 else "#c0504d" for r in drivers["rho"]],
    text=drivers["rho"].round(2), textposition="outside",
    hovertemplate="%{y}<br>correlation: %{x:.2f}<extra></extra>",
))
fig.update_layout(
    title="What goes together with satisfaction? (higher = stronger link)",
    xaxis=dict(range=[-1, 1], title="Correlation with overall satisfaction", zeroline=True),
    height=420, width=900, template="simple_white", margin=dict(l=300),
)
fig.show()

## 7. Accessibility
Only 2 respondents report a difficulty: use as anecdotal evidence only.

In [37]:
display(df[df.difficulty=="Si"][["region","age","digital_skill","used_fse","usability_idx","satisfaction","experience"]])

,region,age,digital_skill,used_fse,usability_idx,satisfaction,experience
33,Puglia,36 - 60 anni,3,Si,1.833333,3.0,NaN
38,Puglia,36 - 60 anni,3,Si,NaN,3.0,Sono riuscita a rilevare l esito degli esami dopo una donazione di sangue


## 8. Open-text coding
Step 1: keyword-based first pass (fast, imperfect). Step 2: **export and review manually** — a human check is needed with such short, varied answers. Edit the dictionary freely.

In [38]:
THEMES = {
  "availability / calendar":  r"disponibil|calendario|date|posti|nessun risultato",
  "map / locations":          r"mappa|sedi|struttur|vicin",
  "login / SPID / CIE":       r"spid|cie\b|accesso|accedere|credenzial|login|password",
  "bugs / errors / speed":    r"bug|non funziona|errore|lent|torna(?:re)? all|ritorna|blocc",
  "clarity / simplicity":     r"chiar|semplic|intuitiv|complicat|confus",
  "reminders / notifications":r"promemoria|notific|avvis|sms|mail|ricorda",
  "app / mobile":             r"\bapp\b|applicazione|smartphone|mobile",
  "documents / reports":      r"referti|document|ricett|prescriz|vaccin|analisi|stor",
  "help / delegate":          r"delega|familiar|aiuto|assisten|anzian|guida|tutorial",
  "payment / tickets":        r"pagament|ticket",
}
def code_text(s):
    s = str(s).lower()
    return [t for t, pat in THEMES.items() if pd.Series([s]).str.contains(pat, regex=True).iloc[0]]

text_cols = {"experience":"experience","useful_features":"useful_features","wishes":"wishes"}
coded = {}
for name, col in text_cols.items():
    t = df[["id","region",col]].dropna()
    t = t[t[col].str.len() > 3]
    t["themes"] = t[col].apply(code_text)
    coded[name] = t
    ex = t.explode("themes").dropna(subset=["themes"]).reset_index(drop=True)
    print(f"\n=== {name}: {len(t)} answers; theme counts by region ===")
    display(pd.crosstab(ex["themes"], ex["region"], margins=True).sort_values("All", ascending=False))
    print("Uncoded answers:", (t["themes"].str.len()==0).sum())


=== experience: 18 answers; theme counts by region ===


region,Emilia-Romagna,Puglia,All
themes,,,
All,4,8,12
clarity / simplicity,1,5,6
availability / calendar,1,3,4
documents / reports,1,0,1
login / SPID / CIE,1,0,1


Uncoded answers: 10

=== useful_features: 16 answers; theme counts by region ===


region,Emilia-Romagna,Puglia,All
themes,,,
All,8,4,12
documents / reports,7,3,10
bugs / errors / speed,0,1,1
login / SPID / CIE,1,0,1


Uncoded answers: 5

=== wishes: 16 answers; theme counts by region ===


region,Emilia-Romagna,Puglia,All
themes,,,
All,14,4,18
clarity / simplicity,4,4,8
availability / calendar,4,0,4
map / locations,3,0,3
login / SPID / CIE,2,0,2
app / mobile,1,0,1


Uncoded answers: 4


In [39]:
# Read the answers behind a theme (pick quotes for the presentation)
def show(name, theme, n=10):
    t = coded[name]
    sub = t[t["themes"].apply(lambda l: theme in l)]
    for _, r in sub.head(n).iterrows(): print(f"[{r.region}] {r[text_cols[name]]}\n")
show("wishes", "availability / calendar")

[Emilia-Romagna] Vorrei che durante la prenotazione di una visita medica mi venga mostrato un calendario delle disponibilità dell'AUSL della mia provincia. Mi sembra strano che mi dica "Nessun risultato per i criteri di ricerca impostati. Premi "Cambia azienda" per cambiare azienda ed eseguire una nuova ricerca".

[Emilia-Romagna] La possibilità di scorrere in un calendario di disponibilità, la visualizzazione delle sedi tramite mappa.

[Emilia-Romagna] Migliore esperienza di utilizzo nel cercare e visualizzare le strutture disponibili e le fasce orarie

[Emilia-Romagna] Tempi più lunghi per completare l'operazione prima che si aggiorni la pagina e maggiore chiarezza relativa alle strutture con disponibilità di prenotazione



In [40]:
# Wishes from Puglia vs competitor, side by side, for manual reading
pd.set_option("display.max_colwidth", 300)
display(coded["wishes"].sort_values("region")[["region","wishes","themes"]])

,region,wishes,themes
0,Emilia-Romagna,"Vorrei che durante la prenotazione di una visita medica mi venga mostrato un calendario delle disponibilità dell'AUSL della mia provincia. Mi sembra strano che mi dica ""Nessun risultato per i criteri di ricerca impostati. Premi ""Cambia azienda"" per cambiare azienda ed eseguire una nuova ricerca"".",[availability / calendar]
1,Emilia-Romagna,Non saprei,[]
5,Emilia-Romagna,"La possibilità di scorrere in un calendario di disponibilità, la visualizzazione delle sedi tramite mappa.","[availability / calendar, map / locations]"
7,Emilia-Romagna,Per stranieri che non abbiamo l’identità digitale ma pagano per avere il ssn possano accedere al fascicolo sanitario elettronico,[login / SPID / CIE]
9,Emilia-Romagna,Maggiore chiarezza,[clarity / simplicity]
10,Emilia-Romagna,Nessuno,[]
16,Emilia-Romagna,Migliore esperienza di utilizzo nel cercare e visualizzare le strutture disponibili e le fasce orarie,"[availability / calendar, map / locations]"
22,Emilia-Romagna,Più semplicita ' nell ' accesso,"[login / SPID / CIE, clarity / simplicity]"
24,Emilia-Romagna,"Miglior adattamento a schermo smartphone, icone troppo piccole, non esageratamente intuitive","[clarity / simplicity, app / mobile]"
34,Emilia-Romagna,Tempi più lunghi per completare l'operazione prima che si aggiorni la pagina e maggiore chiarezza relativa alle strutture con disponibilità di prenotazione,"[availability / calendar, map / locations, clarity / simplicity]"


## 9. Export tables for the report

In [41]:
import os; os.makedirs("outputs", exist_ok=True)
with pd.ExcelWriter("outputs/fse_analysis_tables.xlsx") as xw:
    means.to_excel(xw, sheet_name="likert_means")
    pd.crosstab(df["region"], df["age"]).to_excel(xw, sheet_name="age_by_region")
    adopt.to_excel(xw, sheet_name="adoption")
    pd.crosstab(df["booking_channel"], df["region"]).to_excel(xw, sheet_name="channels")
    for k,v in coded.items(): v.assign(themes=v["themes"].str.join("; ")).to_excel(xw, sheet_name=f"text_{k}", index=False)
print("Saved to outputs/")

Saved to outputs/
